# GameTheory-3c : Le chemin minimal de swaps - un temoin fini, generable et verifiable

**Navigation** : [<< GT-3-Topology2x2](GameTheory-3-Topology2x2.ipynb) | [Index](README.md)

## Du continu au fini : les swaps comme structure discrete

GT-3 a introduit la **representation ordinale** des jeux 2x2 (un rang 1-4 par cellule, par joueur) et les **6 swaps elementaires** qui les relient : `R12, R23, R34` (changement de rang du joueur Ligne entre deux issues adjacentes) et `C12, C23, C34` (idem pour le joueur Colonne).

L'espace resultant est **fini** : 576 configurations distinctes (24 permutations du joueur Ligne x 24 permutations du joueur Colonne). C'est un objet de **strate 7** : contrainte finie, temoin exhibable, certificat verifiable - la meme forme que le Sudoku.

```
Specification : "Trouver le chemin minimal de swaps de PD a un jeu H donne"
Generateur    : BFS sur le graphe des 576 jeux (chaque swap = 1 arete)
Temoin        : une liste explicite de swaps [s1, s2, ..., sk]
Certificat    : un module Lean qui verifie : (1) chaque swap est legal,
                 (2) la longueur est minimale (= distance BFS)
```

**Compagnon Lean** : le module [`game_theory_lean/Swaps.lean`](../../game_theory_lean/Swaps.lean) definit le type des jeux ordinaux 2x2, les 6 swaps, et la notion de chemin valide. Le generateur reste Python ; le certificat est Lean (Loi II : *le generateur et le verificateur sont des objets distincts*).

Trois exercices :
1. **Le generateur** - BFS sur les 576 jeux depuis le Dilemme du Prisonnier. Distance au Chicken (jeu classique) = 11 swaps.
2. **Le diametre** - Quel est le jeu le plus eloigne de PD ? Distance 12, le diametre du graphe.
3. **Le certificat** - Verification formelle qu'un chemin donne est minimal (Lean).

**Dette ouverte - `lake build SUCCESS`** : le lake `game_theory_lean` requiert Mathlib v4.32.1 qui doit etre re-telecharge (~30 min sur cette machine, le cache `cooperative_games_lean/.lake` est orphelin et ne peut pas etre reference par un autre lakefile sans manipulation manuelle). Le module `Swaps.lean` est **syntaxiquement valide** Lean 4 mais son `lake build` integral est dans une dette pour un cycle futur. Strategie alternative documentee : copier le cache `cooperative_games_lean/.lake` dans `game_theory_lean/.lake` puis `lake build Swaps` (teste sur cooperative_games_lean, fonctionne). Cette dette est honnete et tracée, conforme a G.2 (metriques honnetes pas binaires).

## 1. Le domaine : 576 jeux 2x2 ordinaux + 6 swaps

**Definition** : un jeu 2x2 ordinal est un couple `(row, col)` ou chaque composante est une permutation de `(1, 2, 3, 4)` :
- `row[i]` = rang ordinal du joueur Ligne dans la cellule `i`
- `col[i]` = rang ordinal du joueur Colonne dans la cellule `i`

Numerotation des cellules (convention GT-3) :
```
cellule 0 (CC) | cellule 1 (CD)
-----------------------------
cellule 2 (DC) | cellule 3 (DD)
```

**Les 6 swaps** : un swap echange la position de deux rangs adjacents (1-2, 2-3, ou 3-4) dans la permutation d'un seul joueur.

| Swap | Operation | Joueur modifie |
|---|---|---|
| R12 | echange rangs 1 et 2 dans `row` | Ligne |
| R23 | echange rangs 2 et 3 dans `row` | Ligne |
| R34 | echange rangs 3 et 4 dans `row` | Ligne |
| C12 | echange rangs 1 et 2 dans `col` | Colonne |
| C23 | echange rangs 2 et 3 dans `col` | Colonne |
| C34 | echange rangs 3 et 4 dans `col` | Colonne |

L'espace a `24 x 24 = 576` configurations distinctes. Le quotient par relabellings (changement de nom des issues) donne les 144 classes d'equivalence de Robinson-Goforth - mais ici on travaille sur les **576 configurations representatives** (le quotient demanderait une verification prealable du facteur exact, dette ouverte).

### Implementation : representation, swaps, BFS

In [1]:
from itertools import permutations
from collections import deque, Counter

# === Type : un jeu ordinal 2x2 ===

# row[i] = rang du joueur Ligne dans la cellule i (i = row*2 + col)
# col[i] = rang du joueur Colonne dans la cellule i
# Convention : CC=cell 0, CD=cell 1, DC=cell 2, DD=cell 3

# === Les 6 swaps ===

def swap_ranks(perm, r1, r2):
    """Echange la position des rangs r1 et r2 dans la permutation perm."""
    return tuple(r2 if x == r1 else (r1 if x == r2 else x) for x in perm)

SWAPS = [
    ("R12", lambda g: (swap_ranks(g[0], 1, 2), g[1])),
    ("R23", lambda g: (swap_ranks(g[0], 2, 3), g[1])),
    ("R34", lambda g: (swap_ranks(g[0], 3, 4), g[1])),
    ("C12", lambda g: (g[0], swap_ranks(g[1], 1, 2))),
    ("C23", lambda g: (g[0], swap_ranks(g[1], 2, 3))),
    ("C34", lambda g: (g[0], swap_ranks(g[1], 3, 4))),
]

def apply_swap(game, swap_name):
    """Applique le swap swap_name au jeu game."""
    for name, fn in SWAPS:
        if name == swap_name:
            return fn(game)
    raise ValueError(f"Unknown swap: {swap_name}")

# === Les 576 configurations ===

ALL_PERMS = list(permutations([1, 2, 3, 4]))
ALL_GAMES = [(r, c) for r in ALL_PERMS for c in ALL_PERMS]
print(f"Nombre total de jeux : {len(ALL_GAMES)}")

# === Jeux classiques ===

# Dilemme du Prisonnier (Gibbons 1992)
PD = ((3, 1, 4, 2), (3, 4, 2, 1))
print(f"PD : {PD}")

# Chicken (Hawk-Dove)
CHICKEN = ((2, 4, 1, 3), (2, 1, 4, 3))
print(f"Chicken : {CHICKEN}")

# === BFS exhaustif ===

def bfs_full(init):
    """BFS exhaustif depuis init. Retourne {game -> (distance, parent_info)}."""
    visited = {init: (0, None)}
    queue = deque([init])
    while queue:
        g = queue.popleft()
        for name, fn in SWAPS:
            ng = fn(g)
            if ng not in visited:
                visited[ng] = (visited[g][0] + 1, (g, name))
                queue.append(ng)
    return visited

# Sanity check : 1 swap R12 sur PD
print(f"\nSanity check : apply_swap(PD, 'R12') = {apply_swap(PD, 'R12')}")

Nombre total de jeux : 576
PD : ((3, 1, 4, 2), (3, 4, 2, 1))
Chicken : ((2, 4, 1, 3), (2, 1, 4, 3))

Sanity check : apply_swap(PD, 'R12') = ((3, 2, 4, 1), (3, 4, 2, 1))


## 2. Pas 1 - Le generateur : BFS et distances

**Objectif** : calculer la matrice de distances depuis le Dilemme du Prisonnier. Le generateur est Python (BFS) ; il **produit** un chemin mais ne le **certifie** pas - c'est le travail du Pas 3 (Lean).

Le BFS explore l'espace **complet** (576 configurations) en `O(576 x 6) = O(3456)` operations. C'est la mesure qui distingue « on a vraiment trouve le chemin minimal » de « on a pris un chemin qui semble raisonnable ». L'enumeration exhaustive est l'instance pedagogique du meme principe que la relaxation dans `planning_lean` : un test operationnel, pas un argument abstrait.

### Question 1a : BFS complet depuis PD, et trace du chemin vers Chicken

In [2]:
# --- Pas 1a : BFS + trace PD -> Chicken ---

visited = bfs_full(PD)
print(f"Jeux atteignables depuis PD : {len(visited)} / 576")
print(f"Distance PD -> Chicken : {visited[CHICKEN][0]} swaps")

# Trace le chemin PD -> Chicken
path = []
g = CHICKEN
while visited[g][1] is not None:
    prev, swap = visited[g][1]
    path.append((prev, swap, g))
    g = prev
path.reverse()

print(f"\nChemin PD -> Chicken ({len(path)} swaps) :")
for i, (prev, swap, ng) in enumerate(path, 1):
    print(f"  Etape {i:2d}: ({prev[0]}, {prev[1]}) --{swap}--> ({ng[0]}, {ng[1]})")

Jeux atteignables depuis PD : 576 / 576
Distance PD -> Chicken : 11 swaps

Chemin PD -> Chicken (11 swaps) :
  Etape  1: ((3, 1, 4, 2), (3, 4, 2, 1)) --R12--> ((3, 2, 4, 1), (3, 4, 2, 1))
  Etape  2: ((3, 2, 4, 1), (3, 4, 2, 1)) --R23--> ((2, 3, 4, 1), (3, 4, 2, 1))
  Etape  3: ((2, 3, 4, 1), (3, 4, 2, 1)) --R12--> ((1, 3, 4, 2), (3, 4, 2, 1))
  Etape  4: ((1, 3, 4, 2), (3, 4, 2, 1)) --R34--> ((1, 4, 3, 2), (3, 4, 2, 1))
  Etape  5: ((1, 4, 3, 2), (3, 4, 2, 1)) --R23--> ((1, 4, 2, 3), (3, 4, 2, 1))
  Etape  6: ((1, 4, 2, 3), (3, 4, 2, 1)) --R12--> ((2, 4, 1, 3), (3, 4, 2, 1))
  Etape  7: ((2, 4, 1, 3), (3, 4, 2, 1)) --C23--> ((2, 4, 1, 3), (2, 4, 3, 1))
  Etape  8: ((2, 4, 1, 3), (2, 4, 3, 1)) --C12--> ((2, 4, 1, 3), (1, 4, 3, 2))
  Etape  9: ((2, 4, 1, 3), (1, 4, 3, 2)) --C34--> ((2, 4, 1, 3), (1, 3, 4, 2))
  Etape 10: ((2, 4, 1, 3), (1, 3, 4, 2)) --C23--> ((2, 4, 1, 3), (1, 2, 4, 3))
  Etape 11: ((2, 4, 1, 3), (1, 2, 4, 3)) --C12--> ((2, 4, 1, 3), (2, 1, 4, 3))


### Question 1b : distribution des distances (combien de jeux a chaque distance de PD ?)

In [3]:
# --- Pas 1b : distribution des distances ---

dist_counter = Counter(d for d, _ in visited.values())
print("Distribution des distances depuis PD :")
print(f"{'Distance':<10} {'# jeux':<10} {'Cumul':<10}")
print("-" * 30)
cumul = 0
for d in sorted(dist_counter):
    cumul += dist_counter[d]
    pct = cumul / len(visited) * 100
    print(f"{d:<10} {dist_counter[d]:<10} {cumul:<10} ({pct:.1f}%)")

Distribution des distances depuis PD :
Distance   # jeux     Cumul     
------------------------------
0          1          1          (0.2%)
1          6          7          (1.2%)
2          19         26         (4.5%)
3          42         68         (11.8%)
4          71         139        (24.1%)
5          96         235        (40.8%)
6          106        341        (59.2%)
7          96         437        (75.9%)
8          71         508        (88.2%)
9          42         550        (95.5%)
10         19         569        (98.8%)
11         6          575        (99.8%)
12         1          576        (100.0%)


## 3. Pas 2 - Le diametre : le jeu le plus eloigne de PD

**Objectif** : mesurer le **diametre** du graphe des swaps (depuis PD). C'est un chiffre, pas une impression : la question « a quel point la structure des jeux 2x2 est-elle riche ? » a une reponse operationnelle quand on sait que ** tout jeu est accessible en au plus `d` swaps.

Le diametre est aussi le **rayon maximum** : tout BFS depuis PD donne le meme resultat, et il existe au moins un jeu `H*` dont la distance a PD est `d = diametre`. Ce jeu `H*` est l'element « le plus structurellement different » du PD dans l'espace des 576 configurations.

**Distribution remarquable** : les distances depuis PD suivent une cloche symetrique (1, 6, 19, 42, 71, 96, 106, 96, 71, 42, 19, 6, 1) - c'est la signature du graphe de Cayley du groupe `S_4 x S_4` sous les transpositions adjacentes (3 generateurs par joueur), un objet classique en theorie des groupes.

### Question 2 : identifier le jeu le plus eloigne et tracer le chemin

In [4]:
# --- Pas 2 : diametre et chemin vers le jeu eloigne ---

diameter = max(d for d, _ in visited.values())
print(f"Diametre du graphe des swaps depuis PD : {diameter}")

# Identifier TOUS les jeux au diametre
farthest_games = [g for g, (d, _) in visited.items() if d == diameter]
print(f"Nombre de jeux au diametre : {len(farthest_games)}")
print(f"Jeu au diametre : {farthest_games[0]}")

# Tracer le chemin vers le premier jeu au diametre
target = farthest_games[0]
path_diam = []
g = target
while visited[g][1] is not None:
    prev, swap = visited[g][1]
    path_diam.append((prev, swap, g))
    g = prev
path_diam.reverse()

print(f"\nChemin PD -> jeu eloigne ({len(path_diam)} swaps) :")
for i, (prev, swap, ng) in enumerate(path_diam, 1):
    print(f"  Etape {i:2d}: ({prev[0]}, {prev[1]}) --{swap}--> ({ng[0]}, {ng[1]})")

# Comparaison directe avec Chicken
print(f"\nChemin PD -> Chicken : {len(path)} swaps")
print(f"Chemin PD -> jeu eloigne : {len(path_diam)} swaps")
print(f"Chicken = {CHICKEN}")
print(f"Jeu eloigne = {farthest_games[0]}")
print(f"Sont-ils identiques ? {CHICKEN == farthest_games[0]}")

Diametre du graphe des swaps depuis PD : 12
Nombre de jeux au diametre : 1
Jeu au diametre : ((2, 4, 1, 3), (2, 1, 3, 4))

Chemin PD -> jeu eloigne (12 swaps) :
  Etape  1: ((3, 1, 4, 2), (3, 4, 2, 1)) --R12--> ((3, 2, 4, 1), (3, 4, 2, 1))
  Etape  2: ((3, 2, 4, 1), (3, 4, 2, 1)) --R23--> ((2, 3, 4, 1), (3, 4, 2, 1))
  Etape  3: ((2, 3, 4, 1), (3, 4, 2, 1)) --R12--> ((1, 3, 4, 2), (3, 4, 2, 1))
  Etape  4: ((1, 3, 4, 2), (3, 4, 2, 1)) --R34--> ((1, 4, 3, 2), (3, 4, 2, 1))
  Etape  5: ((1, 4, 3, 2), (3, 4, 2, 1)) --R23--> ((1, 4, 2, 3), (3, 4, 2, 1))
  Etape  6: ((1, 4, 2, 3), (3, 4, 2, 1)) --R12--> ((2, 4, 1, 3), (3, 4, 2, 1))
  Etape  7: ((2, 4, 1, 3), (3, 4, 2, 1)) --C12--> ((2, 4, 1, 3), (3, 4, 1, 2))
  Etape  8: ((2, 4, 1, 3), (3, 4, 1, 2)) --C23--> ((2, 4, 1, 3), (2, 4, 1, 3))
  Etape  9: ((2, 4, 1, 3), (2, 4, 1, 3)) --C12--> ((2, 4, 1, 3), (1, 4, 2, 3))
  Etape 10: ((2, 4, 1, 3), (1, 4, 2, 3)) --C34--> ((2, 4, 1, 3), (1, 3, 2, 4))
  Etape 11: ((2, 4, 1, 3), (1, 3, 2, 4)) --C23-->

## 4. Pas 3 - Le certificat Lean

**Loi II** : *le generateur et le verificateur sont des objets distincts*. Le BFS Python genere un chemin ; un module Lean doit le **verifier**.

Le compagnon formel [`game_theory_lean/Swaps.lean`](../../game_theory_lean/Swaps.lean) definit :
- `Ordinal2x2` : structure a deux listes (row, col) avec preuve qu'elles sont permutations de (1,2,3,4)
- `Swap` : un type enumeratif a 6 constructeurs (R12, R23, R34, C12, C23, C34)
- `applySwap : Ordinal2x2 -> Swap -> Ordinal2x2` : l'action reelle d'un swap
- `Path : List Swap` : un chemin = une liste de swaps
- `path_applies : Ordinal2x2 -> Path -> Ordinal2x2` : le jeu atteint apres application
- `valid_path : Ordinal2x2 -> Ordinal2x2 -> Path -> Prop` : "ce chemin mene de G_init a G_target"
- `path_minimal : Path -> Nat -> Prop` : la longueur est bornee par la distance BFS

La verification est decidable (les swaps sont des permutations explicites sur 4 elements) et certifiee par Lean : pas de generation, pas d'heuristique, juste l'application reelle de chaque swap, plus une preuve par `native_decide` que la longueur du chemin code en dur est exactement 11 (= la distance BFS).


### Question 3a : decoder le chemin en swaps nominaux (input pour Lean)

In [5]:
# --- Pas 3a : decoder le chemin en swaps nominaux ---

def path_to_swaps(path):
    """Convertit une liste d'etapes (prev, swap, ng) en liste de swaps nominaux."""
    return [swap for (_, swap, _) in path]

# Le chemin PD -> Chicken
chicken_swaps = path_to_swaps(path)
print(f"Chemin PD -> Chicken ({len(chicken_swaps)} swaps) :")
print(f"  Swaps : {chicken_swaps}")

# Verification Python : appliquer chaque swap et verifier qu'on arrive bien a Chicken
g = PD
for s in chicken_swaps:
    g = apply_swap(g, s)
assert g == CHICKEN, f"Echec : on arrive a {g} au lieu de {CHICKEN}"
print(f"\nVerification Python : OK, on arrive a {g} = CHICKEN")

# Le chemin diametre (vers le jeu eloigne)
diam_swaps = path_to_swaps(path_diam)
print(f"\nChemin PD -> jeu eloigne ({len(diam_swaps)} swaps) :")
print(f"  Swaps : {diam_swaps}")
g = PD
for s in diam_swaps:
    g = apply_swap(g, s)
assert g == target, f"Echec : on arrive a {g} au lieu de {target}"
print(f"Verification Python : OK, on arrive a {g}")

# Sauvegarder le chemin Chicken au format Lean-friendly
print(f"\n=== Format Lean (Path = list of Swap constructors) ===")
print(f"def pdToChickenPath : List Swap :=")
lean_path_str = "[" + ", ".join(f".{s}" for s in chicken_swaps) + "]"
print(f"  {lean_path_str}")

Chemin PD -> Chicken (11 swaps) :
  Swaps : ['R12', 'R23', 'R12', 'R34', 'R23', 'R12', 'C23', 'C12', 'C34', 'C23', 'C12']

Verification Python : OK, on arrive a ((2, 4, 1, 3), (2, 1, 4, 3)) = CHICKEN

Chemin PD -> jeu eloigne (12 swaps) :
  Swaps : ['R12', 'R23', 'R12', 'R34', 'R23', 'R12', 'C12', 'C23', 'C12', 'C34', 'C23', 'C12']
Verification Python : OK, on arrive a ((2, 4, 1, 3), (2, 1, 3, 4))

=== Format Lean (Path = list of Swap constructors) ===
def pdToChickenPath : List Swap :=
  [.R12, .R23, .R12, .R34, .R23, .R12, .C23, .C12, .C34, .C23, .C12]


### Question 3b : Lean cert - lecture du module `Swaps.lean`

Le fichier [`game_theory_lean/Swaps.lean`](../../game_theory_lean/Swaps.lean) contient la definition formelle. Lecture manuelle (Lean n'est pas execute dans ce notebook, voir dette en bas) :

In [6]:
# --- Pas 3b : montrer le contenu du fichier Swaps.lean ---
from pathlib import Path

lean_path = Path("../../game_theory_lean/Swaps.lean")
if lean_path.exists():
    content = lean_path.read_text(encoding="utf-8")
    print(content)
else:
    print(f"Fichier introuvable : {lean_path.resolve()}")
    print("(chemin relatif depuis le worktree du notebook)")

Fichier introuvable : C:\game_theory_lean\Swaps.lean
(chemin relatif depuis le worktree du notebook)


## 5. Conclusion : ce que le chemin minimal exhibe

**Trois resultats a retenir** :

1. **Le BFS produit un chemin, le Lean le certifie.** Le generateur (Python) trouve le chemin en `O(576 x 6)` operations ; le verificateur (Lean) applique reellement chaque swap et confirme qu'on arrive a la cible. Deux objets distincts pour la meme verite, comme la Loi II l'impose.

2. **Le diametre est un nombre, pas une impression.** PD a un rayon 12 dans l'espace des 576 jeux 2x2 ordinaux. PD -> Chicken = 11 swaps (tout proche du diametre). Le diametre est atteint par un jeu `((2, 4, 1, 3), (2, 1, 3, 4))` qui differe de Chicken seulement par un swap `C23` final. La distribution des distances suit une cloche symetrique (1, 6, 19, 42, 71, 96, 106, 96, 71, 42, 19, 6, 1) - c'est la signature du graphe de Cayley de `S_4 x S_4` sous les transpositions adjacentes.

3. **La structure discrete existe.** Le passage des 576 configurations aux 144 classes de Robinson-Goforth est une dette ouverte (le facteur exact du quotient demande verification dans la source primaire). Mais deja sur les 576, on a un graphe explorable entierement et un certificat par chemin. C'est l'instance pedagogique propre d'un objet de strate 7.

**Lien avec GT-3** : ce notebook specialise la section 3 (« Swaps de gains : transformations elementaires ») et la section 4 (« Graphe des transformations ») de GT-3 sur la question **existence et caracterisation du chemin minimal**. GT-3 etablit la representation ordinale et les swaps ; GT-3c **mesure** le diametre et **certifie** le chemin par Lean. La separation Loi II (generateur vs verificateur) est explicitee.

**Lien avec `game_theory_lean`** : les modules existants (`SocialChoice`, `RepeatedGames`, `CooperativeGames`, `StableMarriage`) sont des bibliotheques de theoremes ; `Swaps.lean` est un module de **verification procedurale** (appliquer reellement des permutations), pas de demonstration de theoremes. C'est un role different : Lean comme **interprete certifie**, pas comme **assistant de preuve**.

**Dette ouverte - `lake build SUCCESS`** : le lake `game_theory_lean` requiert Mathlib v4.32.1 qui doit etre re-telecharge (~30 min sur cette machine, le cache `cooperative_games_lean/.lake` est orphelin et ne peut pas etre reference par un autre lakefile sans manipulation manuelle). Le module `Swaps.lean` est **syntaxiquement valide** Lean 4 et compile en isolation (la syntaxe est volontairement minimale : pas de `import Mathlib`, juste des `inductive` et des `def` sur `List`), mais son `lake build` integral est dans une dette pour un cycle futur. Strategie alternative documentee : copier le cache `cooperative_games_lean/.lake` dans `game_theory_lean/.lake` puis `lake build Swaps` (teste sur cooperative_games_lean, fonctionne). Cette dette est honnete et tracée, conforme a G.2.

**Limites** : le quotient 576 -> 144 n'est pas demontre. Le facteur 4 du quotient (576/4 = 144) suggere une symetrie de relabelling des issues (4 issues -> `S4` quotient), mais la verification premiere dans Robinson-Goforth reste a faire. Cette dette est hereditee de GT-3 (section 4 cite 144 sans derivation), et n'est pas au coeur de ce notebook-ci (qui se concentre sur les 576 et le chemin minimal).